# Impedance ML Engineering — Session 06
## Compare trees and select models with grouped CV
**Core: 60 minutes · Classical ML · Optional extension: 20–40 minutes**

## Why this matters
Can nonlinear tabular models outperform ridge without exploiting patient leakage or a large search budget?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Run patient-grouped cross-validation with fold-local preprocessing.
- Compare ridge, random forests, and gradient boosting fairly.
- Distinguish hyperparameter selection from final evaluation.
- Interpret model errors and correlated features cautiously.

## R-to-Python / prerequisite bridge
Grouped resamples → `GroupKFold`; tuning grid → `GridSearchCV`; scores such as `neg_mean_absolute_error` require sign reversal.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Trees partition feature space; forests average randomized trees; boosting adds successive learners to reduce prediction error. Capacity controls such as depth or leaf size alter bias/variance. Flexible models can memorize patient-specific patterns even when patient IDs are excluded.

Grouped CV holds entire patients out while fitting preprocessing within each fold. A search selects hyperparameters from these scores, so the selected CV score can be optimistic. Permutation importance asks how prediction quality changes when a feature is disrupted, conditional on this trained model and evaluation data; correlated features and unrealistic permutations complicate interpretation.

Imports: `GroupKFold`, `GridSearchCV` from `sklearn.model_selection`; `RandomForestRegressor`, `HistGradientBoostingRegressor` from `sklearn.ensemble`; `permutation_importance` from `sklearn.inspection`.

## Task 1 — Audit development folds (6 min)
Combine training and validation indices into development data, reserving test. Create `GroupKFold(n_splits=4)` and print each fold’s patient/row counts. Assert disjoint patient IDs per fold. Predict why patient-equal folds need not have equal row counts. If using supplied partitions, their indices are full-table positions.

## Task 2 — Rebuild the ridge reference (6 min)
Build a fresh numeric/categorical pipeline on development data. Compare alpha 0.1, 1, 10 using `GridSearchCV(...,scoring="neg_mean_absolute_error",cv=cv)` and `.fit(X_dev,y_dev,groups=groups_dev)`. Report positive mean CV MAE. No preprocessing is fitted before CV.

## Task 3 — Add a random forest (8 min)
Use `RandomForestRegressor(n_estimators=100,random_state=42,n_jobs=1)` after preprocessing. Compare max_depth 3 versus None and min_samples_leaf 3 versus 8: four configurations total. Trees need not scale features; retain comparable imputation/category rules. Report train-versus-validation fold errors to inspect overfitting.

## Task 4 — Add gradient boosting (8 min)
Use `HistGradientBoostingRegressor(max_iter=100,early_stopping=False,random_state=42)` with dense preprocessed inputs; compare max_leaf_nodes 7 versus 15, keeping learning_rate=0.05. Disable automatic row-based early stopping for this grouped benchmark. Use identical folds and scores. XGBoost is an optional later comparison rather than another broad search.

## Task 5 — Compare a strong nonfitted baseline (6 min)
Calculate persistence MAE on each validation fold. Assemble ridge/forest/boosting/persistence mean MAE, fold spread, fit time, and model complexity. Explain why fold SD is not automatically a confidence interval: folds share overlapping training data.

## Task 6 — Select without touching test (6 min)
Write a selection rule using CV MAE plus runtime/complexity and choose one candidate. Inspect `best_estimator_` and why it was refitted. Keep selected configuration in `artifacts/06/`. Do not report the original validation rows as independent evaluation after they participated in CV.

## Task 7 — Inspect one model’s held-out fold (8 min)
Refit the selected configuration on one fold’s training portion and examine its fold-validation residuals by horizon/protocol. Use `permutation_importance` on that fold-validation set with 5 repeats and `scoring="neg_mean_absolute_error"`. State that this inspection may motivate future development experiments. Correlated features can split or obscure importance.

## Task 8 — Write the comparison decision (4 min)
Report the selected model, CV scope, search budget, baseline gap, and one failure mode. Explain whether improvement is worth added complexity. Do not require a tree to win or interpret feature importance as a causal mechanism.

## Retrieval practice
Explain why `.fit(X_dev,y_dev)` inside a grouped GridSearchCV still needs `groups=groups_dev`, and what the score sign means.

## Professional reasoning
1. What could go wrong if early stopping silently uses random rows?
2. Why can an extensive search overfit CV results?
3. What extra evaluation would justify deployment at a previously unseen site?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Compare a small XGBRegressor configuration on the same folds, recording library version and runtime. Disable implicit non-grouped validation and keep the same candidate budget.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)